[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ratchanon12/2306565/blob/main/boiler_intro.ipynb)

# What is a database? — hands-on notebook

**Three months of data from a boiler house · February to April 2026**

Something in this boiler house is wrong. Nobody will tell you what, and no single row of data
will show you. By the end of the notebook you will have found it, proved it with two instruments
that do not talk to each other, and be able to say which boiler to open up at the next outage.

You do not need any experience with databases. Run the cells from the top. Where you see
**`# YOUR TURN`** there is a query with **`___`** blanks in it — fill them in and run the cell.
You cannot break anything.

> ### Working in Google Colab? Do this first
> **File → Save a copy in Drive.** That gives you your own copy, saved to your Google
> account, so the answers you type today are still there tomorrow. If you skip it, your work
> disappears when you close the tab.
>
> Run cells with the ▶ button on the left of each cell, or **Shift + Enter**.

### The seven tables

| Table | One row is | Rows |
|---|---|---|
| `boilers` | one boiler — the asset register | 4 |
| `operators` | one person on the shift roster | 6 |
| `readings` | one instrument reading, every 4 hours | 1,866 |
| `daily_logs` | one boiler on one day | 311 |
| `water_tests` | one weekly water sample | 45 |
| `maintenance_events` | one job done on one boiler | 14 |
| `fuel_prices` | what gas cost in one month | 3 |

### Every query you will write today has this shape

```
SELECT   which columns you want
FROM     which table
JOIN     another table  ON  how they match      (only when you need it)
WHERE    which rows you want                    (optional)
GROUP BY what to summarise by                   (optional)
HAVING   which groups to keep                   (optional)
ORDER BY how to sort                            (optional)
LIMIT    how many rows to show                  (optional)
```

Only `SELECT` and `FROM` are compulsory. The clauses must appear in that order.

### How hard is this?

| | |
|---|---|
| Exercise 1 | reading the data — fill in one or two blanks |
| Exercise 2 | summarising — the same, with the blanks further apart |
| Exercise 3 | joining tables — less is filled in, and the last one is all yours |
| Take it further | no answers provided |

## Step 1 — set the database up

Run the next two cells. The first one **builds the database for you** — there is nothing to
upload and nothing to install. The second one connects to it and gives you three tools.

> If Colab disconnects or you come back tomorrow, just run these two cells again.

In [ ]:
# ---- RUN ME FIRST -------------------------------------------------------
# This builds boilerhouse.db. You do not need to read or understand this cell.
import base64, gzip, sqlite3, os

_DATA = """
...
"""

if os.path.exists("boilerhouse.db"):
    os.remove("boilerhouse.db")

_sql = gzip.decompress(base64.b64decode("".join(_DATA.split()))).decode()
_con = sqlite3.connect("boilerhouse.db")
_con.executescript(_sql)
_con.commit()
_con.close()

print("Database ready:", round(os.path.getsize("boilerhouse.db") / 1024), "KB")
print("Now run the next cell.")

In [ ]:
import sqlite3, base64, gzip, json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

con = sqlite3.connect("boilerhouse.db")

_BANK = json.loads(gzip.decompress(base64.b64decode("""
H4sIADw8nGoC/61Y627bNhR+lQMDg5NCUX1JOqDdBuTipW4bZ4jTdgMCqLJE22ol0hWpukFRYA+xJ9yT7PAimZLoOEbnH0FI
ieec7ztX6lun7/c7z+FbZ5lQgf90bpcERDhLCSQcPsxYkpKcf3gBnOUC1olYwvXNxegGzv6CPBQJXQRitYSL0fTc73jQ4Z9T
KWU6ejM6vwWAiMXEAxpm+DcLP5E8yHAn9azDHr6UZQnnCaMkvqO/31xfgfwZ5Xd0m8bOd9TY9wc1AO9fjm5GwJfJXMB4Agfd
s67XPe8eAvz79z/AcgD7jV8BnyOkzfK86wOSQDc450WaBhKCEyHJVgFl3uYtT8uygLAVQcuZhKJ1469uoFR7aAGthBmIwxrE
8+u3k9uDJ4dIXEEFh5yt+QvQuxfj6e14gpZp9oIkrl6Lk/mc5IQK+BKmBeE+nDH054JBQkGg3zkqBA3MAbXS2vqdToEGOQlj
dA737qjZ3m6QOlC51/BUCjCQj2uQK2a4CKNPgZCsRyoIPGk6hTfjq/EtnDhdVGnGd7lXE2F5qdRvuaGl7I5qPfg7MXaeOKKv
UogBNYDTyQXqRQtfj6A76A2eHfWOj3r9n7oqInnytVLtAaYAkC8kv4c5K3JY4h/uwPSkxVoVWjXd0hdSvVu/QfDMlT828l/g
pOfDG8Y+QShUpGglENJYLeNQEAUGF/cQ5gTCNK1CyoP1MomWsqDIrQhjzwGp5SQSZro8OP3Vwt2yWKEbNMrb6bvLg0r0IQY+
T2ICNxipFwe+73vQP/RhwuASt/7AEHiuPEKLbIZw51g9tHOMdgcMLaquRkqVMZ+RkAbVtjvwB41yVmWdVC7pw2TXCe0p+ktD
Lb8XNCY5JaFY7siGKlOhZvaGx6bl8klkH9teExpVwZlnbdsNBcNWS+JrPCIj6Or0z7qJcARX40ljT8biOk+EjFQscfOCRgLb
C4dwEWKxM26XdCrg+/LUUljyFLE0Jlw4TjTNLk8smRCPO+ECWkpR9OzHcb2+XoYcsEuBYJSq1j99e3UwL0gaLD4ewlO1NJGr
2RVMhKnsKV+QyRhm92YHmz0TGJkQYqKECwJsrto24/uSbKtUONvxpVSWGeWWUGJ4UIJ56WEJDRY8GCrqFx8DJC5QxFkeiMMk
vQ9S5vaB1WPs85upZtDoKzIH1IyEWbBmeSxd9PL03Xhy6cNYwDxJBfZSWOSsWOEsoGtiuSvng12NcY8C8Mg4u6PaQEl0Uyz8
Bv2qQtf7TyUIuQtkYzENviJMZq1yG2CqRYRKA1Qp1K156ES6EbYluGQSYb0og2mXJ0t59rBgHW8OC0MFddhoRq+ucf4zYxDM
4BoXvt2+883K7T5fz5u5Lztm7u8YbCC/o0qlNVzvUGvP3r6Ro7YrZP2egTZo12wSMXTLRyYrLt9MmHwZrnBC0DvzJOfiOSi7
qjkZmLSK+eWGtiu211sI0fcN5lvTOPPVrO3h+U0UxP5DjobYwRS0uQJllcVWdWgfKBbJ2kQ3wfWm+EqyqsjDOSCOwTG2nY1u
349GExyF1Ax43Oth3kCcs5ViHpugyItM3gfmYZHKEFKJtslkxeZDRDsHiHoc/tAIYYaIgtfuhT8Uz9UFrG6mgyy7pim4lp9s
MJuiPWw01amMdoxzbLYezAqhU6G0tD69aa5hKq/ZVl3jTPkKSz4XMokY3tTY6sHg3xS42G/0rirqH9+9nKngILmWBtuJc3e7
YaPblWFUzelPUZJ1+X+CWYE3kkvZ7eToodUoRmvvGfb0DkQ4lcxk+WFr+siotqVtj/banL8tjOvD/25hTsxlMq0iEbB5oB//
X6nR8JpX02/5UD82jnvW7mj71L5mzTHF2umespA7p7Q9Y/3xFahVjfQksCtV9msATfwW22bH0P1zuwlkeKMRhIY0IgHeTuWX
HizzxumGX/NZQqeJ/OZ1NOj3hvJLF/kqgBISc5CzZUrgc8HwQuJ0QOYr+aaFlitxv1Ir9a0iiDG55IpKKRZNDiuzx8Vo5izf
LSx2dbbs7Hz//h/BxvI65RQAAA==
""".replace("\n", ""))).decode())


def q(sql):
    """Run a SQL query and hand back the answer as a table."""
    try:
        return pd.read_sql_query(sql, con)
    except Exception as e:
        print("SQL did not run:", str(e).strip().splitlines()[-1].split(": ", 1)[-1])
        if "___" in sql:
            print("(there is still a ___ blank in that query)")
        return pd.DataFrame()


def _rows(df):
    """Turn a table into something comparable: row order and column order do not matter."""
    out = []
    for _, row in df.iterrows():
        cells = []
        for v in row:
            try:
                cells.append(("n", float(v)))
            except (TypeError, ValueError):
                cells.append(("s", str(v).strip().lower()))
        out.append(sorted(cells, key=lambda c: (c[0], c[1])))
    return sorted(out, key=repr)


def _same(got, want):
    if len(got) != len(want):
        return False
    for a, b in zip(got, want):
        if len(a) != len(b):
            return False
        for (ta, va), (tb, vb) in zip(a, b):
            if ta != tb:
                return False
            if ta == "n" and abs(va - vb) > max(0.06, 0.005 * abs(vb)):
                return False
            if ta == "s" and va != vb:
                return False
    return True


def check(tag, result):
    """Show your answer, then say whether it is right. Column names are ignored."""
    display(result)
    want = pd.read_sql_query(_BANK[tag]["sql"], con)
    if result is None or len(result) == 0:
        print(f"[{tag}] nothing to check yet — the answer has "
              f"{len(want)} row(s) and {want.shape[1]} column(s).")
    elif _same(_rows(result), _rows(want)):
        print(f"[{tag}] correct — {len(want)} row(s), {want.shape[1]} column(s).  OK")
    else:
        print(f"[{tag}] not there yet. The answer has {len(want)} row(s) x {want.shape[1]} "
              f"column(s); you have {len(result)} x {result.shape[1]}.")
        print(f"      Try  hint('{tag}')  — or  solution('{tag}')  once you have really tried.")


def hint(tag):
    print(_BANK[tag]["hint"])


def solution(tag):
    print(_BANK[tag]["sql"])


# A first look. Change 'boilers' to any other table name and run it again.
q("SELECT * FROM boilers")

That is the whole mechanism: you write SQL between the quotes, and `q(...)` hands you back a
table. Three tools go with it:

* `check("1.1", q("""..."""))` — runs your query, shows the table, and tells you whether it is right.
  It ignores column names, column order and row order; it cares about the values.
* `hint("1.1")` — one line of help.
* `solution("1.1")` — one correct answer. Use it *after* you have tried, not instead.

**Two rules that catch everybody:**

* Text goes in single quotes — `code = 'B-2103'` is right, `code = "B-2103"` is not.
* Numbers do not — `boiler_id = 2`, `rating_tph > 30`.

In [ ]:
# The other small tables. Six people, three shifts — and what the gas cost.
display(q("SELECT * FROM operators"))
display(q("SELECT * FROM fuel_prices"))

In [ ]:
# The first few rows of the big tables, and the whole maintenance log.
display(q("SELECT * FROM readings    LIMIT 5"))
display(q("SELECT * FROM daily_logs  LIMIT 5"))
display(q("SELECT * FROM water_tests LIMIT 5"))
display(q("SELECT * FROM maintenance_events"))

> **Look carefully at `readings`.** It never names a boiler — it says `boiler_id = 2`. That
> number is an internal id, and it is **not** the tag painted on the boiler: `boiler_id = 2` is
> tag `B-2103`, not `B-2102`. The only way to know which machine a row belongs to is to look the
> id up in `boilers`.
>
> Storing the boiler's tag, name, maker and rating *once*, in one place, and pointing at them
> from everywhere else, is the whole idea of a database. Guessing from the id is exactly the
> habit this notebook is trying to break. We put the two back together in Exercise 3.

> **And read the maintenance log now.** Fourteen rows, in plain English, written by the people
> who did the work. Most of what you are about to discover is explained in there — but only if
> you already know which boiler to look up.

---
# Exercise 1 · Reading the data

`SELECT` · `WHERE` · `AND` / `OR` · `ORDER BY` · `LIMIT` · `COUNT` · `DISTINCT`

**Worked example first.** Pick some columns, keep some rows, sort the result:

In [ ]:
q("""
SELECT   code, name, rating_tph
FROM     boilers
WHERE    rating_tph > 30
ORDER BY code
""")

### 1.1 The asset register: `code`, `name`, `maker_model`, `rating_tph`, `commissioned` — biggest first.

Then answer out loud: what does one row represent? **Which two of the four are sister units** —
identical machines that can fairly be compared with each other? And which one is too new and too
small to compare with anything?

In [ ]:
# YOUR TURN 1.1
check("1.1", q("""
SELECT   code, name, maker_model, rating_tph, commissioned
FROM     ___
ORDER BY ___ DESC
"""))

### 1.2 The operators on shift **B or C** — `emp_no`, `full_name` and `shift`, in alphabetical order.

Either `shift IN ('B', 'C')` or `shift = 'B' OR shift = 'C'`. Note that `shift = 'B' OR 'C'`
is *not* SQL, even though it is how you would say it out loud.

In [ ]:
# YOUR TURN 1.2
check("1.2", q("""
SELECT   emp_no, full_name, shift
FROM     operators
WHERE    ___
ORDER BY ___
"""))

### 1.3 In one row: how many readings are there, and how many different boilers do they cover?

Two aggregates side by side in the same `SELECT`. *Predict the first number before you run it:*
4 boilers × 6 readings a day × 89 days would be 2,136. It is not. Two of the four boilers have
fewer rows than the others, for two completely different reasons — 2.2 shows you, and the
maintenance log in 3.7 explains it.

In [ ]:
# YOUR TURN 1.3
check("1.3", q("""
SELECT COUNT(*) AS n_readings,
       ___      AS n_boilers
FROM   readings
"""))

### 1.4 The five hottest stack temperatures — `boiler_id`, `ts`, `stack_temp_c`, hottest first.

**Now look at the top row like an engineer, not like a computer.** Flue gas at that temperature
would have destroyed the boiler. Is that a boiler telling you something, or an instrument telling
you something? Which four rows would you actually trust?

In [ ]:
# YOUR TURN 1.4
check("1.4", q("""
SELECT   boiler_id, ts, stack_temp_c
FROM     readings
ORDER BY ___
LIMIT    ___
"""))

### 1.5 Every reading for `boiler_id = 2` taken on 1 April 2026 — all columns.

The `ts` column holds a date *and* a time, so match the start of the text:
`ts LIKE '2026-04-01%'`. Join the two conditions with `AND`.

In [ ]:
# YOUR TURN 1.5
check("1.5", q("""
SELECT *
FROM   readings
WHERE  boiler_id = 2
  AND  ___
"""))

### 1.6 The readings that cannot be true

A boiler making steam cannot have flue gas colder than the water in it. Find every reading
where `stack_temp_c` is **below 50 °C** — show `boiler_id`, `ts`, `steam_tph` and `stack_temp_c`.

Then read what you get: same boiler, same day, and the steam flow beside it looks perfectly
normal. **The boiler was working. Only the thermocouple was dead.** Remember this one — you will
need it in Exercise 3, and the maintenance log knows about it.

In [ ]:
# YOUR TURN 1.6
check("1.6", q("""
SELECT boiler_id, ts, steam_tph, stack_temp_c
FROM   readings
WHERE  ___
"""))

---
# Exercise 2 · Summarising

`AVG` · `MIN` · `MAX` · `SUM` · `COUNT` · `GROUP BY` · `HAVING`

**Worked example.** One number out of 1,866 rows — and then one number *per boiler*:

In [ ]:
display(q("""
SELECT ROUND(AVG(steam_tph), 1) AS mean_steam_tph
FROM   readings
"""))

display(q("""
SELECT   boiler_id,
         ROUND(AVG(steam_tph), 1) AS mean_steam_tph,
         COUNT(*)                 AS n_readings
FROM     readings
GROUP BY boiler_id
"""))

`GROUP BY boiler_id` tells the database to deal the rows into piles — one pile per boiler —
and answer your question separately for each pile.

`COUNT(*)` is there as a check. The four counts are **not** equal, and the reason is not a
mistake in the data. Keep it in the corner of your eye.

### 2.1 The average steam flow across every reading, to one decimal place.

In [ ]:
# YOUR TURN 2.1
check("2.1", q("""
SELECT ROUND(___, 1) AS mean_steam_tph
FROM   readings
"""))

### 2.2 The average **stack temperature** for each boiler, with the row count beside it.

**Look hard at the answer.** One boiler runs far hotter than the others — say how many degrees
out loud. And one boiler has far fewer readings than the others; make a note to find out why.

In [ ]:
# YOUR TURN 2.2
check("2.2", q("""
SELECT   boiler_id,
         ROUND(AVG(stack_temp_c), 1) AS mean_stack_c,
         ___                         AS n_readings
FROM     readings
GROUP BY ___
"""))

### 2.3 For each boiler: the lowest, the highest, and the **swing** between them.

The swing is `MAX(...) - MIN(...)`.

Two of these four swings are impossible for a working boiler. You already know what both of them
are — 1.4 found one and 1.6 found the other. **An average never shows you this. A minimum and a
maximum do, and they cost nothing.** Get into the habit.

In [ ]:
# YOUR TURN 2.3
check("2.3", q("""
SELECT   boiler_id,
         ROUND(MIN(stack_temp_c), 1) AS coldest,
         ROUND(MAX(stack_temp_c), 1) AS hottest,
         ROUND(___ - ___, 1)         AS swing
FROM     readings
GROUP BY boiler_id
"""))

### 2.4 From `daily_logs`: total steam, total gas, and **gas per tonne of steam** for each boiler — worst first.

Gas per tonne is `SUM(fuel_gj) / SUM(steam_t)`.

> **Why not `AVG(fuel_gj / steam_t)`?** Because that treats a quiet 400-tonne day and a flat-out
> 700-tonne day as equally important. Totals divided by totals weights each day by how much steam
> it actually made. The two answers differ, and only one of them is the plant's real fuel bill.

In [ ]:
# YOUR TURN 2.4
check("2.4", q("""
SELECT   boiler_id,
         ROUND(SUM(steam_t), 1) AS total_steam_t,
         ROUND(SUM(fuel_gj), 1) AS total_fuel_gj,
         ROUND(___ / ___, 3)    AS gj_per_tonne
FROM     daily_logs
GROUP BY boiler_id
ORDER BY gj_per_tonne DESC
"""))

### 2.5 Which boilers have an average stack temperature above 150 °C?

`WHERE` throws away **rows before** they are grouped. To throw away **whole groups** you need
`HAVING`, which goes after `GROUP BY`. `WHERE AVG(stack_temp_c) > 150` is an error — try it if
you like, the message is worth seeing.

One word goes in the blank.

In [ ]:
# YOUR TURN 2.5
check("2.5", q("""
SELECT   boiler_id, ROUND(AVG(stack_temp_c), 1) AS mean_stack_c
FROM     readings
GROUP BY boiler_id
___      AVG(stack_temp_c) > 150
"""))

### 2.6 The three busiest days on the site

Groups do not have to be boilers. Group `daily_logs` by `log_date` instead, total the steam from
every boiler, and show the three biggest days.

In [ ]:
# YOUR TURN 2.6
check("2.6", q("""
SELECT   log_date, ROUND(SUM(steam_t), 1) AS site_steam_t
FROM     daily_logs
GROUP BY ___
ORDER BY ___ DESC
LIMIT    3
"""))

**A picture of what you just found.** Run this cell once you have done 2.2:

In [ ]:
daily = q("""
SELECT   substr(r.ts, 1, 10) AS day,
         b.code,
         AVG(r.stack_temp_c) AS stack_c
FROM     readings r
JOIN     boilers b ON b.boiler_id = r.boiler_id
WHERE    r.stack_temp_c BETWEEN 50 AND 400
GROUP BY day, b.code
ORDER BY day
""")

colours = {"B-2101": "#1C7293", "B-2102": "#0B3C49", "B-2103": "#E07A3F", "B-2104": "#7BA05B"}
fig, ax = plt.subplots(figsize=(11, 3.8))
for code_, colour in colours.items():
    part = daily[daily["code"] == code_]
    ax.plot(pd.to_datetime(part["day"]), part["stack_c"], lw=1.6, color=colour, label=code_)
ax.set_ylabel("stack temperature, °C")
ax.set_title("Daily average stack temperature, February to April")
ax.legend(loc="upper left", ncol=4)
ax.grid(alpha=.3)
plt.tight_layout()
plt.show()

Three things in one picture, and you should be able to say all three out loud:

1. one line climbs steadily for three months and never comes back;
2. one line climbs through February and then **falls off a cliff in March** — something was done
   to that boiler, and it worked;
3. one line starts in the middle of March, because that boiler did not exist before then.

Keep the cliff in mind. Exercise 3 ends at the table that explains it.

---
# Exercise 3 · Joining tables

`JOIN ... ON`

`readings` only knows `boiler_id = 2`. A **JOIN** is how you look that `2` up in the
`boilers` table and glue the two rows together.

**Worked example.** Read the `ON` line as a sentence: *where these two numbers match, that is
the same boiler.*

In [ ]:
q("""
SELECT b.name, r.ts, r.stack_temp_c
FROM   readings r
JOIN   boilers  b  ON  b.boiler_id = r.boiler_id
LIMIT  10
""")

The letters `r` and `b` are just nicknames, so you can write `r.ts` instead of
`readings.ts`. You choose them yourself.

### 3.1 The first 10 readings with the boiler's **name** instead of its number.

In [ ]:
# YOUR TURN 3.1
check("3.1", q("""
SELECT   b.name, r.ts, r.stack_temp_c
FROM     readings r
JOIN     ___ b  ON  ___
ORDER BY r.reading_id
LIMIT    10
"""))

### 3.2 The first 10 daily logs with the boiler **code**, the **operator's name** and their **shift**.

Two JOINs — one to `boilers`, one to `operators`. The second is the same line as the first,
written again with different names.

In [ ]:
# YOUR TURN 3.2
check("3.2", q("""
SELECT   b.code, o.full_name, o.shift, d.log_date, d.steam_t
FROM     daily_logs d
JOIN     boilers b  ON  b.boiler_id = d.boiler_id
JOIN     ___     o  ON  ___
ORDER BY d.log_id
LIMIT    10
"""))

### 3.3 Average stack temperature per boiler **code** — this time without the broken readings.

In 2.2 you averaged everything, including a dead thermocouple reading 0 °C and a spike of 511 °C.
Do it again with `WHERE r.stack_temp_c BETWEEN 50 AND 400`, and show `COUNT(*)` so you can see
how many rows survived.

Compare the numbers with 2.2. One boiler moves by more than a degree. **That is what six bad
rows out of 1,866 can do** — and nobody would ever have noticed from the average alone.

In [ ]:
# YOUR TURN 3.3
check("3.3", q("""
SELECT   b.code,
         ROUND(AVG(r.stack_temp_c), 1) AS mean_stack_c,
         COUNT(*)                      AS n_used
FROM     readings r
JOIN     boilers b ON ___
WHERE    ___
GROUP BY ___
ORDER BY mean_stack_c DESC
"""))

### 3.4 Gas per tonne of steam per boiler **code**, worst at the top.

This is 2.4 again with a join on top. Nothing is filled in for you this time.

**Which boiler is worst, and by how much?** Compare it with its sister unit — the one with the
same maker, the same rating and the same year — not with the site average.

In [ ]:
# YOUR TURN 3.4
check("3.4", q("""
SELECT   ___
FROM     daily_logs d
___
"""))

### 3.5 Is the bad boiler simply being worked harder?

A fair comparison needs **load**, not just output. For each boiler show `rating_tph`, the average
`steam_tph`, and the average as a **percentage of the rating**:
`AVG(r.steam_tph) / b.rating_tph * 100`.

This one matters. If the hot boiler were running flat out, a high stack temperature would be
normal and there would be nothing to investigate. Is it?

In [ ]:
# YOUR TURN 3.5
check("3.5", q("""
SELECT   b.code,
         b.rating_tph,
         ROUND(AVG(r.steam_tph), 1) AS mean_steam_tph,
         ROUND(___, 1)              AS pct_of_rating
FROM     readings r
JOIN     boilers b ON b.boiler_id = r.boiler_id
GROUP BY b.code, b.rating_tph
"""))

### 3.6 Rule out the people

Maybe one shift simply runs the plant badly. For each **shift** show gas per tonne
(`SUM(d.fuel_gj) / SUM(d.steam_t)`) and how many logs it covers.

If the three numbers come out the same, you have eliminated a suspect. Eliminating a suspect is
a result, even though the table looks boring.

In [ ]:
# YOUR TURN 3.6
check("3.6", q("""
SELECT   o.shift,
         ROUND(SUM(d.fuel_gj) / SUM(d.steam_t), 3) AS gj_per_tonne,
         COUNT(*)                                  AS n_logs
FROM     daily_logs d
JOIN     ___ o ON ___
GROUP BY ___
"""))

### 3.7 Ask the people who were there

Every job done on a boiler is in `maintenance_events`. Show `event_date`, `event_type`,
`hours_down` and `notes` for the boiler you named in 3.4 — **by its tag, not its id**, so the
query says what you mean: `WHERE b.code = 'B-2103'`.

Four rows. Read the notes. One of them tried the obvious fix and reports that it did not work;
one of them, right at the end, is somebody else noticing what you have just spent an hour proving.

Then run the same query for `'B-2101'` — the boiler whose line fell off a cliff in March — and
read the date next to `'Tube cleaning'`. **That is your cause, and your fix, in a table nobody
thinks of as data.**

In [ ]:
# YOUR TURN 3.7
check("3.7", q("""
SELECT   m.event_date, m.event_type, m.hours_down, m.notes
FROM     maintenance_events m
JOIN     ___ b ON ___
WHERE    ___
ORDER BY m.event_date
"""))

### 3.8 No query for this one — write the finding

You have everything you need. In the cell below, write **four sentences**:

1. Which boiler, and what you think is physically wrong with it.
2. The two independent numbers that support you, and which table each came from.
3. What you would do about it, and what the maintenance log says happened when somebody did
   exactly that to another boiler in March.
4. What would change your mind — what result would tell you that you are wrong?

The last sentence is the one engineers skip and the one that matters. An explanation that
nothing could disprove is not an explanation.

In [ ]:
answer = """

1. ...
2. ...
3. ...
4. ...

"""
print(answer)

---
## Check yourself

Run this once you have finished. It works out the numbers, compares them with the rule of thumb
engineers use — **a boiler loses roughly 1 % of its efficiency for every 20 °C of extra flue gas
temperature** — and then goes one step past the exercises to show you what the same three months
look like month by month.

You are not asked to write these queries. Read them if you are curious: they are the ones in the
*Take it further* list at the bottom.

In [ ]:
level = q("""
SELECT   b.code,
         ROUND(AVG(r.stack_temp_c), 1) AS mean_stack_c
FROM     readings r
JOIN     boilers b ON b.boiler_id = r.boiler_id
WHERE    r.stack_temp_c BETWEEN 50 AND 400
GROUP BY b.code
""")
fuel = q("""
SELECT   b.code, ROUND(SUM(d.fuel_gj) / SUM(d.steam_t), 3) AS gj_per_tonne
FROM     daily_logs d
JOIN     boilers b ON b.boiler_id = d.boiler_id
GROUP BY b.code
""")
both = level.merge(fuel, on="code")
display(both)

sisters = both[both["code"].isin(["B-2101", "B-2103"])].set_index("code")
d_temp = sisters.loc["B-2103", "mean_stack_c"] - sisters.loc["B-2101", "mean_stack_c"]
d_fuel = sisters.loc["B-2103", "gj_per_tonne"] / sisters.loc["B-2101", "gj_per_tonne"] - 1

print("\n1) THE LEVEL — two identical boilers, side by side")
print(f"   B-2103 runs {d_temp:.0f} °C hotter at the stack than its sister B-2101.")
print(f"   B-2103 burns {d_fuel*100:.1f} % more gas per tonne of steam.")
print(f"   Rule of thumb: {d_temp:.0f} °C / 20 = {d_temp/20:.1f} % efficiency lost.")
print(f"   Measured from the fuel figures:      {d_fuel*100:.1f} %")

trend = q("""
SELECT   substr(r.ts, 1, 7) AS month, AVG(r.stack_temp_c) AS stack_c
FROM     readings r
WHERE    r.boiler_id = 2 AND r.stack_temp_c BETWEEN 50 AND 400
GROUP BY month
ORDER BY month
""")
rise = trend["stack_c"].iloc[-1] - trend["stack_c"].iloc[0]
print("\n2) THE TREND")
print(f"   B-2103 climbed {rise:.0f} °C in three months — about {rise/3:.0f} °C a month,")
print(f"   and it has not stopped.")

fixed = q("""
SELECT   substr(r.ts, 1, 7) AS month, AVG(r.stack_temp_c) AS stack_c
FROM     readings r
WHERE    r.boiler_id = 1 AND r.stack_temp_c BETWEEN 50 AND 400
GROUP BY month
ORDER BY month
""")
print("\n3) THE PROOF THAT CLEANING WORKS")
print(f"   B-2101 went {fixed['stack_c'].iloc[0]:.0f} °C → {fixed['stack_c'].iloc[-1]:.0f} °C "
      f"across the same three months —")
print("   because it was given a tube cleaning on 2026-03-10. Same make, same rating, same fuel.")

money = q("""
SELECT (SUM(fuel_gj) - SUM(steam_t) *
        (SELECT SUM(fuel_gj) / SUM(steam_t) FROM daily_logs WHERE boiler_id = 1)) *
       (SELECT AVG(baht_per_gj) FROM fuel_prices) AS extra_baht
FROM   daily_logs
WHERE  boiler_id = 2
""")
print("\n4) THE MONEY")
print(f"   {money['extra_baht'].iloc[0]/1e6:.2f} million baht of gas in three months, "
      f"burnt for nothing.")
print("\nTwo instruments, two tables, a maintenance log and a price list.")
print("That is what evidence looks like — and the fix is already proven on the boiler next to it.")

**And the same three months, one row per month.** This is the picture the numbers above came
from — and the one query beyond today's exercises that would have produced it:

In [ ]:
mn_temp = q("""
SELECT   substr(r.ts, 1, 7) AS month, b.code, AVG(r.stack_temp_c) AS stack_c
FROM     readings r
JOIN     boilers b ON b.boiler_id = r.boiler_id
WHERE    r.stack_temp_c BETWEEN 50 AND 400
GROUP BY month, b.code
ORDER BY month
""")

mn_fuel = q("""
SELECT   substr(d.log_date, 1, 7) AS month, b.code,
         SUM(d.fuel_gj) / SUM(d.steam_t) AS gj_per_tonne
FROM     daily_logs d
JOIN     boilers b ON b.boiler_id = d.boiler_id
GROUP BY month, b.code
ORDER BY month
""")

colours = {"B-2101": "#1C7293", "B-2102": "#0B3C49", "B-2103": "#E07A3F", "B-2104": "#7BA05B"}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))
for code_, colour in colours.items():
    part = mn_temp[mn_temp["code"] == code_]
    ax1.plot(part["month"], part["stack_c"], "o-", color=colour, lw=2, label=code_)
    part = mn_fuel[mn_fuel["code"] == code_]
    ax2.plot(part["month"], part["gj_per_tonne"], "o-", color=colour, lw=2, label=code_)

ax1.set_title("Stack temperature by month")
ax1.set_ylabel("°C")
ax2.set_title("Gas per tonne of steam by month")
ax2.set_ylabel("GJ / t")
for ax in (ax1, ax2):
    ax.grid(alpha=.3)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print("B-2101 was given a tube cleaning on 2026-03-10. Both instruments noticed.")

---
## What you learned today

1. **A table is a list of one kind of thing.** If you cannot say what one row is in four
   words, it should probably be two tables.
2. **Store every fact once** and point at it from everywhere else. That is why a database
   stays correct while a spreadsheet slowly drifts.
3. **`GROUP BY` is the one to remember.** Turning 1,866 rows into one number per boiler is what
   found the problem today.
4. **Choose the group deliberately.** Grouped by boiler you get a level. Group the same rows by
   *month* instead — one line of SQL — and you get a trend, which is usually the more valuable
   answer.
5. **Check `MIN` and `MAX` before you trust an `AVG`.** Six dead rows and one spike were sitting
   in the data the whole time, and only the extremes showed them.
6. **The boring tables win the argument.** The maintenance log is fourteen rows of typing, and it
   supplied the cause, the fix, and the date the fix worked.
7. **The computer counts, you interpret.** SQL gave you 170 °C, 3.07 GJ per tonne and a date in
   March. Only an engineer can turn that into *"clean the tubes at the next outage."*

### Take it further — no answers provided

1. **Group by month.** `substr(ts, 1, 7)` cuts a timestamp down to `'2026-03'`. Put it in
   `SELECT` and in `GROUP BY`, and you get the two pictures printed above — the trend, not the
   level. Do it for both `readings` and `daily_logs` and check that they agree.
2. **Put a price on it.** `fuel_prices` has the gas price for each month. Join it on the month you
   cut out yourself — `ON f.month = substr(d.log_date, 1, 7)` — and work out what B-2103's extra
   gas cost the plant this quarter.
3. **Compare against a number you have not worked out yet.** A query in brackets returns one
   value: `WHERE stack_temp_c > (SELECT AVG(stack_temp_c) FROM readings)`. How many readings from
   each boiler are above the site average?
4. **Was the burner service a red herring?** Compare B-2103's stack temperature in the two weeks
   before and after 18 March. Does the note in the log match the data?
3. **How long does a cleaning last?** B-2101 was cleaned on 10 March. Fit its stack temperature
   since then and estimate when it will need doing again.
4. **The water angle.** Is the silica in `water_tests` higher on the fouling boiler? Is five
   samples a month enough to say so?
5. **The new unit.** B-2104 has the best gas per tonne on site. Before you recommend running it
   harder, find two reasons from `boilers` why that comparison is not fair.
6. **Your own question.** Ask the database something nobody set you, and check whether the answer
   would survive somebody disagreeing with it.

### Keeping your work

* **Your answers** are already saved if you did *File → Save a copy in Drive* at the start.
  If you did not, do it now — **File → Save a copy in Drive**.
* **The database file** is rebuilt by the first cell every time, so you never need to keep it.
  But if you would like a copy to open in DB Browser for SQLite at home, run the cell below.

In [ ]:
# Optional: download boilerhouse.db to your own computer (Colab only).
try:
    from google.colab import files
    files.download("boilerhouse.db")
except ImportError:
    print("Not running in Colab — the file is already in this folder:",
          os.path.abspath("boilerhouse.db"))

In [ ]:
con.close()
print("Done. Try asking the database a question nobody set you.")